<a href="https://colab.research.google.com/github/shaestasaleem/flyrank-machine-learning-internship/blob/main/work/notebooks/w06_validation_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/shaestasaleem/flyrank-machine-learning-internship/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

### Finding 1 — Growth prediction

The FlyRank research paper reports that its growth-prediction model achieved about 90% accuracy on unseen pages from the same brands and about 75% accuracy on completely unseen brands.

**My methodology question:**  
How exactly was the growing-versus-declining label created, and were all input features restricted to information available before the outcome period? Because growth is a time-based outcome, I would also ask how performance changes under a strictly time-aware validation design rather than relying only on repeated random splits.

**Why I would ask this:**  
The difference between same-brand and unseen-brand performance is useful because it shows that generalizing to a completely new brand is harder. I would want to understand how much of the remaining performance comes from stable page-level patterns and how much may depend on brand-specific or time-specific patterns.

I am not treating the result as wrong. My question is whether the label timeline and validation design support the strength and scope of the claim.


### Finding 2 — Refresh ROI

The paper reports that refreshed pages showed positive impression lift and that 7 of 9 held-out strata showed statistically significant refresh lift.

**My methodology question:**  
How were refreshed pages selected and compared with stale pages, and how were differences that existed before the refresh controlled? For example, were the two groups comparable in previous visibility, content age, search demand, and the reason a page was selected for refreshing?

**Why I would ask this:**  
Pages chosen for refresh may already be different from pages that are left unchanged. A team might choose pages with stronger historical traffic or clearer opportunity. I would therefore want to separate the measured association after a refresh from a causal claim that the refresh itself produced the improvement.

For my own capstone, this reminds me to prefer language such as “observed,” “measured,” and “associated with” unless the validation design supports a stronger causal statement.

In [1]:
import pandas as pd

paper_audit = pd.DataFrame({
    "finding": [
        "Growth prediction",
        "Refresh ROI"
    ],
    "reported_result": [
        "90% same-brand unseen pages; 75% unseen brands",
        "7 of 9 held-out strata showed significant refresh lift"
    ],
    "methodology_focus": [
        "Label timeline + time-aware/generalization validation",
        "Refresh selection + comparability/confounding"
    ]
})

print("Two FlyRank paper findings selected for methodology review:")
display(paper_audit)

Two FlyRank paper findings selected for methodology review:


,finding,reported_result,methodology_focus
0,Growth prediction,90% same-brand unseen pages; 75% unseen brands,Label timeline + time-aware/generalization val...
1,Refresh ROI,7 of 9 held-out strata showed significant refr...,Refresh selection + comparability/confounding


## 2. My model under an honest split (before/after)

## 2. My model under an honest split (before/after)

For this audit, I compare the same Logistic Regression model under two validation designs.

**Before:** a random row split. This is easier, but rows from the same client can appear in both training and validation data. Because pages from the same client may share hidden patterns, this can make performance look stronger than it would on a completely unseen client.

**After:** grouped validation by `client_hash_id`. All rows from one client stay entirely in either training or validation for a fold.

I keep the model, features, target, and evaluation metrics the same. The only important change is the validation design.

My main ranking metrics are Precision@20 and Precision@50, with ROC-AUC as a broader ranking measure. I also print the base rate so the model score is not interpreted without context.

All decision-time features are available by March 24, 2026. The March 25–31 window is used only to define the future outcome.

In [2]:
%pip -q install duckdb huggingface_hub pandas numpy scikit-learn

import duckdb
import numpy as np
import pandas as pd
import sklearn

from google.colab import userdata
from IPython.display import display

from sklearn.model_selection import StratifiedKFold, GroupKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

RANDOM_SEED = 42
N_SPLITS = 5

# ------------------------------------------------------------
# Connect to the FlyRank warehouse
# ------------------------------------------------------------

hf_token = userdata.get("HF_TOKEN")

if not hf_token:
    raise ValueError(
        "HF_TOKEN not found. Add it in Colab Secrets "
        "and enable notebook access."
    )

con = duckdb.connect()

con.execute("INSTALL httpfs;")
con.execute("LOAD httpfs;")

safe_token = hf_token.replace("'", "''")

con.execute(f"""
CREATE OR REPLACE SECRET hf_token (
    TYPE HUGGINGFACE,
    TOKEN '{safe_token}'
);
""")

WAREHOUSE = "hf://datasets/FlyRank/internship-warehouse"

MARCH_DAILY = (
    f"{WAREHOUSE}/fact_content_daily_performance/"
    "month=2026-03/data_0.parquet"
)

print("Warehouse connection ready.")
print("Development month: March 2026")
print("June 2026 remains sealed.")
print("Random seed:", RANDOM_SEED)
print("scikit-learn:", sklearn.__version__)


# ------------------------------------------------------------
# Rebuild the same Week-5 modeling frame
# ------------------------------------------------------------

signal_frame = con.sql(f"""
WITH daily AS (
    SELECT
        report_date,
        client_hash_id,
        content_hash_id,
        gsc_impressions,
        gsc_clicks,
        gsc_sum_position
    FROM read_parquet('{MARCH_DAILY}')
    WHERE gsc_data_available IS TRUE
),

windows AS (
    SELECT
        client_hash_id,
        content_hash_id,

        COUNT(DISTINCT report_date) AS available_days,

        SUM(
            CASE
                WHEN report_date BETWEEN DATE '2026-03-11'
                                     AND DATE '2026-03-17'
                THEN COALESCE(gsc_impressions, 0)
                ELSE 0
            END
        ) AS previous_impressions_7d,

        SUM(
            CASE
                WHEN report_date BETWEEN DATE '2026-03-11'
                                     AND DATE '2026-03-17'
                THEN COALESCE(gsc_clicks, 0)
                ELSE 0
            END
        ) AS previous_clicks_7d,

        SUM(
            CASE
                WHEN report_date BETWEEN DATE '2026-03-18'
                                     AND DATE '2026-03-24'
                THEN COALESCE(gsc_impressions, 0)
                ELSE 0
            END
        ) AS recent_impressions_7d,

        SUM(
            CASE
                WHEN report_date BETWEEN DATE '2026-03-18'
                                     AND DATE '2026-03-24'
                THEN COALESCE(gsc_clicks, 0)
                ELSE 0
            END
        ) AS recent_clicks_7d,

        SUM(
            CASE
                WHEN report_date BETWEEN DATE '2026-03-18'
                                     AND DATE '2026-03-24'
                THEN COALESCE(gsc_sum_position, 0)
                ELSE 0
            END
        ) AS recent_sum_position_7d,

        SUM(
            CASE
                WHEN report_date BETWEEN DATE '2026-03-25'
                                     AND DATE '2026-03-31'
                THEN COALESCE(gsc_clicks, 0)
                ELSE 0
            END
        ) AS future_clicks_7d

    FROM daily

    GROUP BY
        client_hash_id,
        content_hash_id
)

SELECT *
FROM windows

WHERE available_days = 31
  AND previous_impressions_7d > 0
  AND recent_impressions_7d > 0
""").df()


# ------------------------------------------------------------
# Decision-time features
# ------------------------------------------------------------

signal_frame["recent_ctr"] = (
    signal_frame["recent_clicks_7d"]
    / signal_frame["recent_impressions_7d"]
)

signal_frame["recent_avg_position"] = (
    signal_frame["recent_sum_position_7d"]
    / signal_frame["recent_impressions_7d"]
)

signal_frame["pre_decision_click_change_7d"] = (
    signal_frame["recent_clicks_7d"]
    - signal_frame["previous_clicks_7d"]
)

# Outcome only
signal_frame["future_click_decline"] = (
    signal_frame["future_clicks_7d"]
    < signal_frame["recent_clicks_7d"]
).astype(int)


# Same clean Week-5 population
model_frame = signal_frame[
    (signal_frame["recent_impressions_7d"] >= 20)
    & (signal_frame["recent_clicks_7d"] > 0)
].copy()

model_frame["position_bucket"] = pd.cut(
    model_frame["recent_avg_position"],
    bins=[0, 3, 10, 20, np.inf],
    labels=["1-3", "4-10", "11-20", "20+"],
    include_lowest=True
)

TARGET = "future_click_decline"

BASE_FEATURES = [
    "recent_impressions_7d",
    "recent_clicks_7d",
    "recent_ctr",
    "recent_avg_position",
    "pre_decision_click_change_7d",
]

FEATURES = BASE_FEATURES + ["ctr_vs_expected"]

print("\nModeling rows:", f"{len(model_frame):,}")
print(
    "Overall future-decline base rate:",
    round(model_frame[TARGET].mean(), 4)
)


# ------------------------------------------------------------
# Metric helper
# ------------------------------------------------------------

def precision_at_k(y_true, scores, k):
    if len(y_true) < k:
        return np.nan

    order = np.argsort(np.asarray(scores))[::-1][:k]

    return np.asarray(y_true)[order].mean()


# ------------------------------------------------------------
# Evaluate one validation design
# ------------------------------------------------------------

def evaluate_split(splitter, split_name, use_groups=False):

    fold_rows = []

    split_iterator = (
        splitter.split(
            model_frame,
            y=model_frame[TARGET],
            groups=model_frame["client_hash_id"]
        )
        if use_groups
        else splitter.split(
            model_frame,
            model_frame[TARGET]
        )
    )

    for fold, (train_idx, test_idx) in enumerate(
        split_iterator,
        start=1
    ):

        train_df = model_frame.iloc[train_idx].copy()
        test_df = model_frame.iloc[test_idx].copy()

        # ----------------------------------------------------
        # Expected CTR learned ONLY from training rows
        # ----------------------------------------------------

        position_ctr = (
            train_df
            .groupby(
                "position_bucket",
                observed=True
            )["recent_ctr"]
            .median()
        )

        overall_train_ctr = train_df["recent_ctr"].median()

        train_df["expected_ctr_for_position"] = (
            train_df["position_bucket"]
            .map(position_ctr)
            .astype(float)
            .fillna(overall_train_ctr)
        )

        test_df["expected_ctr_for_position"] = (
            test_df["position_bucket"]
            .map(position_ctr)
            .astype(float)
            .fillna(overall_train_ctr)
        )

        train_df["ctr_vs_expected"] = (
            train_df["recent_ctr"]
            / train_df["expected_ctr_for_position"].replace(0, np.nan)
        )

        test_df["ctr_vs_expected"] = (
            test_df["recent_ctr"]
            / test_df["expected_ctr_for_position"].replace(0, np.nan)
        )

        assert not train_df[FEATURES].isna().any().any()
        assert not test_df[FEATURES].isna().any().any()

        # ----------------------------------------------------
        # Check client overlap
        # ----------------------------------------------------

        train_clients = set(train_df["client_hash_id"])
        test_clients = set(test_df["client_hash_id"])

        client_overlap = len(
            train_clients.intersection(test_clients)
        )

        # ----------------------------------------------------
        # Train same Logistic Regression
        # ----------------------------------------------------

        model = Pipeline([
            ("scale", StandardScaler()),
            (
                "logistic",
                LogisticRegression(
                    max_iter=2000,
                    random_state=RANDOM_SEED
                )
            )
        ])

        X_train = train_df[FEATURES]
        y_train = train_df[TARGET]

        X_test = test_df[FEATURES]
        y_test = test_df[TARGET]

        model.fit(X_train, y_train)

        scores = model.predict_proba(X_test)[:, 1]

        fold_rows.append({
            "validation_design": split_name,
            "fold": fold,
            "test_rows": len(test_df),
            "test_clients": len(test_clients),
            "client_overlap": client_overlap,
            "base_rate": y_test.mean(),
            "precision_at_20": precision_at_k(
                y_test,
                scores,
                20
            ),
            "precision_at_50": precision_at_k(
                y_test,
                scores,
                50
            ),
            "roc_auc": roc_auc_score(
                y_test,
                scores
            )
        })

    return pd.DataFrame(fold_rows)


# ------------------------------------------------------------
# BEFORE: random row validation
# ------------------------------------------------------------

random_cv = StratifiedKFold(
    n_splits=N_SPLITS,
    shuffle=True,
    random_state=RANDOM_SEED
)

random_results = evaluate_split(
    random_cv,
    split_name="Random row split",
    use_groups=False
)


# ------------------------------------------------------------
# AFTER: grouped client validation
# ------------------------------------------------------------

group_cv = GroupKFold(
    n_splits=N_SPLITS
)

group_results = evaluate_split(
    group_cv,
    split_name="Grouped by client",
    use_groups=True
)


# ------------------------------------------------------------
# Fold-level results
# ------------------------------------------------------------

all_results = pd.concat(
    [
        random_results,
        group_results
    ],
    ignore_index=True
)

print("\nFold-level validation results")
display(all_results.round(4))


# ------------------------------------------------------------
# Before / after summary
# ------------------------------------------------------------

validation_comparison = (
    all_results
    .groupby("validation_design")
    .agg(
        mean_base_rate=("base_rate", "mean"),
        mean_precision_at_20=("precision_at_20", "mean"),
        mean_precision_at_50=("precision_at_50", "mean"),
        mean_roc_auc=("roc_auc", "mean"),
        std_roc_auc=("roc_auc", "std"),
        max_client_overlap=("client_overlap", "max")
    )
    .reset_index()
)

print("\nBefore / after validation comparison")
display(validation_comparison.round(4))


# ------------------------------------------------------------
# Explicit validation check
# ------------------------------------------------------------

print("\nRandom split maximum client overlap:",
      int(random_results["client_overlap"].max()))

print("Grouped split maximum client overlap:",
      int(group_results["client_overlap"].max()))

assert group_results["client_overlap"].max() == 0

print(
    "\nPASS: grouped validation keeps clients "
    "completely separated."
)


Warehouse connection ready.
Development month: March 2026
June 2026 remains sealed.
Random seed: 42
scikit-learn: 1.6.1


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


Modeling rows: 27,186
Overall future-decline base rate: 0.5504

Fold-level validation results


,validation_design,fold,test_rows,test_clients,client_overlap,base_rate,precision_at_20,precision_at_50,roc_auc
0,Random row split,1,5438,25,25,0.5504,0.75,0.82,0.6759
1,Random row split,2,5437,29,28,0.5503,0.90,0.92,0.6704
2,Random row split,3,5437,25,25,0.5503,0.65,0.78,0.6681
3,Random row split,4,5437,26,25,0.5503,0.85,0.80,0.6634
4,Random row split,5,5437,27,27,0.5505,0.85,0.88,0.6572
5,Grouped by client,1,7020,1,0,0.5528,0.75,0.90,0.7078
6,Grouped by client,2,5695,1,0,0.5645,0.90,0.92,0.7035
7,Grouped by client,3,4926,1,0,0.5568,0.80,0.76,0.6473
8,Grouped by client,4,4773,14,0,0.5839,0.85,0.84,0.6975
9,Grouped by client,5,4772,14,0,0.4895,0.75,0.78,0.5886



Before / after validation comparison


,validation_design,mean_base_rate,mean_precision_at_20,mean_precision_at_50,mean_roc_auc,std_roc_auc,max_client_overlap
0,Grouped by client,0.5495,0.81,0.84,0.6689,0.0511,0
1,Random row split,0.5504,0.80,0.84,0.6670,0.0071,28



Random split maximum client overlap: 28
Grouped split maximum client overlap: 0

PASS: grouped validation keeps clients completely separated.


### Before/after interpretation

The random row split allowed substantial client overlap between training and validation. In some folds, as many as 28 clients appeared on both sides of the split. The grouped validation removed this overlap completely.

However, the grouped result did not become weaker. Average Precision@20 was 0.79 with the random split and 0.81 with grouped client validation. Precision@50 was 0.84 under both designs, while ROC-AUC was 0.6673 for the random split and 0.6689 for the grouped split.

This means I did not observe evidence that client overlap was materially inflating the average performance of this Logistic Regression model in this March experiment.

The grouped folds were much more variable, though. ROC-AUC standard deviation increased from 0.0067 under random splitting to 0.0511 under grouped validation. I interpret this as evidence that performance differs across unseen client groups, so the grouped result is still the more honest validation design even though its average score is similar.

I therefore keep the grouped-client validation as my main result and treat the random split only as an audit comparison.

## 3. Leakage audit

## 3. Leakage audit

I audited the final Week-5 feature set against three leakage risks: label-derived information, future or overlapping windows, and decision-derived product signals.

My model features are:

- `recent_impressions_7d`
- `recent_clicks_7d`
- `recent_ctr`
- `recent_avg_position`
- `pre_decision_click_change_7d`
- `ctr_vs_expected`

These features use information available no later than March 24, 2026.

The outcome `future_click_decline` is created from March 25–31 clicks and is used only as the label. `future_clicks_7d` is therefore forbidden as a model feature. Client and content hash IDs are also excluded from the model; `client_hash_id` is used only for grouped validation.

`ctr_vs_expected` requires extra care. In grouped validation, the expected CTR for each position bucket is calculated from training clients only and then applied to held-out clients.

I found one population-selection limitation. My modeling frame requires `available_days == 31`. Because that condition checks whether an item has data throughout the whole month, it indirectly depends on availability during the March 25–31 outcome window. I do not treat this as a model feature, but it means the evaluated population is restricted to content with full-month availability. I disclose this limitation rather than describing the population as fully decision-time selected.

As a verification test, I deliberately add a copy of the target as a feature. If the leakage test is working correctly, performance should jump toward a perfect score. I then remove that feature and keep only the honest result.

In [3]:
# ============================================================
# SECTION 3 — Leakage audit
# ============================================================

from sklearn.model_selection import GroupKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

# ------------------------------------------------------------
# 1. Feature audit
# ------------------------------------------------------------

feature_audit = pd.DataFrame([
    {
        "field": "recent_impressions_7d",
        "window_or_role": "March 18-24",
        "allowed_as_feature": True,
        "reason": "Known before the March 24 decision point"
    },
    {
        "field": "recent_clicks_7d",
        "window_or_role": "March 18-24",
        "allowed_as_feature": True,
        "reason": "Known before the March 24 decision point"
    },
    {
        "field": "recent_ctr",
        "window_or_role": "Derived from March 18-24",
        "allowed_as_feature": True,
        "reason": "Uses only recent decision-time clicks and impressions"
    },
    {
        "field": "recent_avg_position",
        "window_or_role": "Derived from March 18-24",
        "allowed_as_feature": True,
        "reason": "Known at the decision point"
    },
    {
        "field": "pre_decision_click_change_7d",
        "window_or_role": "March 11-17 vs March 18-24",
        "allowed_as_feature": True,
        "reason": "Both windows occur before the outcome"
    },
    {
        "field": "ctr_vs_expected",
        "window_or_role": "Decision-time derived",
        "allowed_as_feature": True,
        "reason": "Expected CTR is learned from training clients only"
    },
    {
        "field": "future_clicks_7d",
        "window_or_role": "March 25-31",
        "allowed_as_feature": False,
        "reason": "Outcome-window information"
    },
    {
        "field": "future_click_decline",
        "window_or_role": "Label",
        "allowed_as_feature": False,
        "reason": "This is the target itself"
    },
    {
        "field": "client_hash_id",
        "window_or_role": "Identifier / grouping",
        "allowed_as_feature": False,
        "reason": "Used only for grouped validation"
    },
    {
        "field": "content_hash_id",
        "window_or_role": "Identifier",
        "allowed_as_feature": False,
        "reason": "Identifier, not a predictive feature"
    }
])

print("Feature leakage audit")
display(feature_audit)


# ------------------------------------------------------------
# 2. Programmatic forbidden-feature check
# ------------------------------------------------------------

FORBIDDEN_FEATURES = {
    "future_clicks_7d",
    "future_click_decline",
    "client_hash_id",
    "content_hash_id"
}

used_forbidden = sorted(
    set(FEATURES).intersection(FORBIDDEN_FEATURES)
)

print("\nFinal model features:")
print(FEATURES)

print("\nForbidden fields found in FEATURES:")
print(used_forbidden)

assert len(used_forbidden) == 0, (
    "Leakage risk: forbidden fields appear in the feature list."
)

print("PASS: no forbidden outcome or identifier fields are model inputs.")


# ------------------------------------------------------------
# 3. Population-selection audit
# ------------------------------------------------------------

population_audit = pd.DataFrame([
    {
        "selection_rule": "previous_impressions_7d > 0",
        "decision_time_safe": True,
        "note": "Uses March 11-17 only"
    },
    {
        "selection_rule": "recent_impressions_7d > 0",
        "decision_time_safe": True,
        "note": "Uses March 18-24 only"
    },
    {
        "selection_rule": "recent_clicks_7d > 0",
        "decision_time_safe": True,
        "note": "Uses March 18-24 only"
    },
    {
        "selection_rule": "recent_impressions_7d >= 20",
        "decision_time_safe": True,
        "note": "Uses March 18-24 only"
    },
    {
        "selection_rule": "available_days == 31",
        "decision_time_safe": False,
        "note": (
            "Requires full-month availability, including "
            "the March 25-31 outcome window"
        )
    }
])

print("\nPopulation-selection audit")
display(population_audit)


# ------------------------------------------------------------
# 4. Deliberate leakage test
# ------------------------------------------------------------
# Add an explicit copy of the target.
# This is intentionally WRONG and exists only to verify
# that the audit harness detects leakage.

leak_test_frame = model_frame.copy()

leak_test_frame["DELIBERATE_LABEL_LEAK"] = (
    leak_test_frame[TARGET]
)

HONEST_BASE_FEATURES = [
    "recent_impressions_7d",
    "recent_clicks_7d",
    "recent_ctr",
    "recent_avg_position",
    "pre_decision_click_change_7d",
]

LEAK_FEATURE = "DELIBERATE_LABEL_LEAK"

gkf = GroupKFold(n_splits=5)

leakage_results = []

for fold, (train_idx, test_idx) in enumerate(
    gkf.split(
        leak_test_frame,
        y=leak_test_frame[TARGET],
        groups=leak_test_frame["client_hash_id"]
    ),
    start=1
):

    train_df = leak_test_frame.iloc[train_idx].copy()
    test_df = leak_test_frame.iloc[test_idx].copy()

    # Expected CTR learned only from training clients
    position_ctr = (
        train_df
        .groupby(
            "position_bucket",
            observed=True
        )["recent_ctr"]
        .median()
    )

    overall_train_ctr = train_df["recent_ctr"].median()

    train_df["expected_ctr_for_position"] = (
        train_df["position_bucket"]
        .map(position_ctr)
        .astype(float)
        .fillna(overall_train_ctr)
    )

    test_df["expected_ctr_for_position"] = (
        test_df["position_bucket"]
        .map(position_ctr)
        .astype(float)
        .fillna(overall_train_ctr)
    )

    train_df["ctr_vs_expected"] = (
        train_df["recent_ctr"]
        / train_df[
            "expected_ctr_for_position"
        ].replace(0, np.nan)
    )

    test_df["ctr_vs_expected"] = (
        test_df["recent_ctr"]
        / test_df[
            "expected_ctr_for_position"
        ].replace(0, np.nan)
    )

    honest_features = (
        HONEST_BASE_FEATURES
        + ["ctr_vs_expected"]
    )

    leaky_features = (
        honest_features
        + [LEAK_FEATURE]
    )

    y_train = train_df[TARGET]
    y_test = test_df[TARGET]

    # Honest model
    honest_model = Pipeline([
        ("scale", StandardScaler()),
        (
            "logistic",
            LogisticRegression(
                max_iter=2000,
                random_state=RANDOM_SEED
            )
        )
    ])

    honest_model.fit(
        train_df[honest_features],
        y_train
    )

    honest_scores = honest_model.predict_proba(
        test_df[honest_features]
    )[:, 1]

    # Deliberately leaky model
    leaky_model = Pipeline([
        ("scale", StandardScaler()),
        (
            "logistic",
            LogisticRegression(
                max_iter=2000,
                random_state=RANDOM_SEED
            )
        )
    ])

    leaky_model.fit(
        train_df[leaky_features],
        y_train
    )

    leaky_scores = leaky_model.predict_proba(
        test_df[leaky_features]
    )[:, 1]

    leakage_results.append({
        "fold": fold,
        "base_rate": y_test.mean(),

        "honest_precision_at_20": precision_at_k(
            y_test,
            honest_scores,
            20
        ),

        "leaky_precision_at_20": precision_at_k(
            y_test,
            leaky_scores,
            20
        ),

        "honest_roc_auc": roc_auc_score(
            y_test,
            honest_scores
        ),

        "leaky_roc_auc": roc_auc_score(
            y_test,
            leaky_scores
        )
    })


leakage_results = pd.DataFrame(
    leakage_results
)

print("\nDeliberate leakage test by grouped fold")
display(leakage_results.round(4))


leakage_summary = pd.DataFrame({
    "model": [
        "Honest feature set",
        "With deliberate label leak"
    ],
    "mean_precision_at_20": [
        leakage_results[
            "honest_precision_at_20"
        ].mean(),

        leakage_results[
            "leaky_precision_at_20"
        ].mean()
    ],
    "mean_roc_auc": [
        leakage_results[
            "honest_roc_auc"
        ].mean(),

        leakage_results[
            "leaky_roc_auc"
        ].mean()
    ]
})

print("\nHonest vs deliberately leaky model")
display(leakage_summary.round(4))


# ------------------------------------------------------------
# 5. Harness sanity check
# ------------------------------------------------------------

honest_auc = (
    leakage_results["honest_roc_auc"].mean()
)

leaky_auc = (
    leakage_results["leaky_roc_auc"].mean()
)

print(
    "\nHonest grouped ROC-AUC:",
    round(honest_auc, 4)
)

print(
    "Leaky grouped ROC-AUC:",
    round(leaky_auc, 4)
)

assert leaky_auc > 0.95, (
    "The deliberate leak did not produce a near-perfect score. "
    "Inspect the validation harness."
)

print(
    "PASS: deliberate leakage creates an obviously "
    "unrealistic score, so the audit harness detects it."
)


Feature leakage audit


,field,window_or_role,allowed_as_feature,reason
0,recent_impressions_7d,March 18-24,True,Known before the March 24 decision point
1,recent_clicks_7d,March 18-24,True,Known before the March 24 decision point
2,recent_ctr,Derived from March 18-24,True,Uses only recent decision-time clicks and impr...
3,recent_avg_position,Derived from March 18-24,True,Known at the decision point
4,pre_decision_click_change_7d,March 11-17 vs March 18-24,True,Both windows occur before the outcome
5,ctr_vs_expected,Decision-time derived,True,Expected CTR is learned from training clients ...
6,future_clicks_7d,March 25-31,False,Outcome-window information
7,future_click_decline,Label,False,This is the target itself
8,client_hash_id,Identifier / grouping,False,Used only for grouped validation
9,content_hash_id,Identifier,False,"Identifier, not a predictive feature"



Final model features:
['recent_impressions_7d', 'recent_clicks_7d', 'recent_ctr', 'recent_avg_position', 'pre_decision_click_change_7d', 'ctr_vs_expected']

Forbidden fields found in FEATURES:
[]
PASS: no forbidden outcome or identifier fields are model inputs.

Population-selection audit


,selection_rule,decision_time_safe,note
0,previous_impressions_7d > 0,True,Uses March 11-17 only
1,recent_impressions_7d > 0,True,Uses March 18-24 only
2,recent_clicks_7d > 0,True,Uses March 18-24 only
3,recent_impressions_7d >= 20,True,Uses March 18-24 only
4,available_days == 31,False,"Requires full-month availability, including th..."



Deliberate leakage test by grouped fold


,fold,base_rate,honest_precision_at_20,leaky_precision_at_20,honest_roc_auc,leaky_roc_auc
0,1,0.5528,0.75,1.0,0.7078,1.0
1,2,0.5645,0.90,1.0,0.7035,1.0
2,3,0.5568,0.80,1.0,0.6473,1.0
3,4,0.5839,0.85,1.0,0.6975,1.0
4,5,0.4895,0.75,1.0,0.5886,1.0



Honest vs deliberately leaky model


,model,mean_precision_at_20,mean_roc_auc
0,Honest feature set,0.81,0.6689
1,With deliberate label leak,1.00,1.0000



Honest grouped ROC-AUC: 0.6689
Leaky grouped ROC-AUC: 1.0
PASS: deliberate leakage creates an obviously unrealistic score, so the audit harness detects it.


In [4]:
# ------------------------------------------------------------
# Real failure examples from grouped out-of-fold predictions
# ------------------------------------------------------------

oof_rows = []

gkf = GroupKFold(n_splits=5)

for fold, (train_idx, test_idx) in enumerate(
    gkf.split(
        model_frame,
        y=model_frame[TARGET],
        groups=model_frame["client_hash_id"]
    ),
    start=1
):

    train_df = model_frame.iloc[train_idx].copy()
    test_df = model_frame.iloc[test_idx].copy()

    # Learn expected CTR from training clients only
    position_ctr = (
        train_df
        .groupby("position_bucket", observed=True)["recent_ctr"]
        .median()
    )

    overall_train_ctr = train_df["recent_ctr"].median()

    train_df["expected_ctr_for_position"] = (
        train_df["position_bucket"]
        .map(position_ctr)
        .astype(float)
        .fillna(overall_train_ctr)
    )

    test_df["expected_ctr_for_position"] = (
        test_df["position_bucket"]
        .map(position_ctr)
        .astype(float)
        .fillna(overall_train_ctr)
    )

    train_df["ctr_vs_expected"] = (
        train_df["recent_ctr"]
        / train_df["expected_ctr_for_position"].replace(0, np.nan)
    )

    test_df["ctr_vs_expected"] = (
        test_df["recent_ctr"]
        / test_df["expected_ctr_for_position"].replace(0, np.nan)
    )

    model = Pipeline([
        ("scale", StandardScaler()),
        (
            "logistic",
            LogisticRegression(
                max_iter=2000,
                random_state=RANDOM_SEED
            )
        )
    ])

    model.fit(
        train_df[FEATURES],
        train_df[TARGET]
    )

    test_df["model_score"] = model.predict_proba(
        test_df[FEATURES]
    )[:, 1]

    test_df["predicted_class"] = (
        test_df["model_score"] >= 0.5
    ).astype(int)

    test_df["fold"] = fold

    # Keep only public-safe fields
    oof_rows.append(
        test_df[
            [
                "fold",
                TARGET,
                "model_score",
                "recent_impressions_7d",
                "recent_clicks_7d",
                "recent_ctr",
                "recent_avg_position",
                "pre_decision_click_change_7d",
                "ctr_vs_expected"
            ]
        ].copy()
    )

oof = pd.concat(oof_rows, ignore_index=True)

false_positives = (
    oof[
        (oof[TARGET] == 0)
        & (oof["model_score"] >= 0.5)
    ]
    .sort_values("model_score", ascending=False)
    .head(3)
)

false_negatives = (
    oof[
        (oof[TARGET] == 1)
        & (oof["model_score"] < 0.5)
    ]
    .sort_values("model_score", ascending=True)
    .head(3)
)

print("Three confident false positives")
display(false_positives.round(4))

print("\nThree confident false negatives")
display(false_negatives.round(4))

print(
    "\nFalse positives:",
    ((oof[TARGET] == 0) & (oof["model_score"] >= 0.5)).sum()
)

print(
    "False negatives:",
    ((oof[TARGET] == 1) & (oof["model_score"] < 0.5)).sum()
)

Three confident false positives


,fold,future_click_decline,model_score,recent_impressions_7d,recent_clicks_7d,recent_ctr,recent_avg_position,pre_decision_click_change_7d,ctr_vs_expected
26172,5,0,0.9999,91.0,9.0,0.0989,3.4505,-2.0,27.8407
20022,4,0,0.9998,64.0,6.0,0.0938,3.6406,1.0,25.1250
18976,4,0,0.9998,42.0,3.0,0.0714,39.9048,-2.0,31.3571



Three confident false negatives


,fold,future_click_decline,model_score,recent_impressions_7d,recent_clicks_7d,recent_ctr,recent_avg_position,pre_decision_click_change_7d,ctr_vs_expected
23785,5,1,0.0000,3805.0,65.0,0.0171,2.6176,-175.0,4.7220
24518,5,1,0.0000,25230.0,248.0,0.0098,2.6789,-131.0,2.7171
23186,5,1,0.0002,946.0,26.0,0.0275,6.4461,-116.0,7.7368



False positives: 6003
False negatives: 4429



## 4. Claim rewrite

### Too strong

> My model identifies which content pages need to be refreshed.

This goes beyond my evidence. My label measures whether clicks decline in the following seven-day window. It does not directly measure whether a page needs a refresh, and my analysis does not test whether refreshing a page causes performance to improve.

### Evidence-safe version

> In the March 2026 development experiment, Logistic Regression ranked future-click-decline outcomes more effectively than my Week-4 rule baseline under grouped client validation. Average Precision@20 was 0.81, compared with 0.67 for the baseline. I treat the model score as a directional decision-support signal for prioritizing editorial review, not as proof that a page needs to be refreshed.

### Why I rewrote it

The revised claim says what was actually measured, names the validation setting, compares against the baseline, and limits the practical interpretation.

It avoids saying that the model predicts a true need for refresh or proves that a refresh would improve performance. Those questions would require additional evidence and a stronger validation or causal design.

In [5]:
claim_audit = pd.DataFrame({
    "item": [
        "Observed target",
        "Validation design",
        "Model Precision@20",
        "Week-4 baseline Precision@20",
        "Supported practical use",
        "Not supported"
    ],
    "evidence": [
        "Future 7-day click decline",
        "5-fold grouped validation by client",
        0.81,
        0.67,
        "Directional prioritization for editorial review",
        "Proof that a page needs refresh or that refresh causes improvement"
    ]
})

print("Evidence supporting the rewritten claim:")
display(claim_audit)

Evidence supporting the rewritten claim:


,item,evidence
0,Observed target,Future 7-day click decline
1,Validation design,5-fold grouped validation by client
2,Model Precision@20,0.81
3,Week-4 baseline Precision@20,0.67
4,Supported practical use,Directional prioritization for editorial review
5,Not supported,Proof that a page needs refresh or that refres...


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.